# Hypergraph Native Adapter v1

目标：把 `hypergraph_mse` 从 grouped feature Ridge proxy 推进到 `src/model` 的 native HGNN-style hypergraph convolution adapter。Notebook 只保留实验流程、外部参考 URL 和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。


In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import HypergraphGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'hypergraph_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse', 'hypergraph_mse', 'thgnn_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])
estimator, constant_prediction = fit_estimator_for_model('hypergraph_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('hypergraph_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, HypergraphGraphRegressor)
assert estimator.feature_count_ == 8
assert estimator.node_count_ == 8
assert estimator.hyperedge_count_ >= 4
assert estimator.incidence_matrix_.shape == (8, estimator.hyperedge_count_)
assert estimator.hypergraph_propagation_.shape == (8, 8)
assert hasattr(estimator, 'model_')
assert capability['paper_fidelity'] == 'native_graph'
assert capability['is_proxy'] is False
assert model_adapter_name('hypergraph_mse') == 'hgnn_hypergraph_convolution_mse_v1'
assert 'hypergraph_mse' in fidelity['native_models']
assert 'hypergraph_mse' not in fidelity['proxy_models']
assert len(fidelity['native_models']) == 25
assert len(fidelity['proxy_models']) == 7

comparison_row = outputs.model_comparison.loc[outputs.model_comparison['model'].eq('hypergraph_mse')].iloc[0].to_dict()
ledger_row = outputs.model_run_ledger.loc[outputs.model_run_ledger['model'].eq('hypergraph_mse')].iloc[0].to_dict()
summary = {
    'schema_version': 'research.hypergraph_native_adapter_v1',
    'status': 'completed',
    'model': 'hypergraph_mse',
    'adapter': model_adapter_name('hypergraph_mse'),
    'estimator': estimator.__class__.__name__,
    'node_count': estimator.node_count_,
    'feature_count': estimator.feature_count_,
    'hyperedge_count': estimator.hyperedge_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'matrix_models': sorted(outputs.predictions['model'].astype(str).unique().tolist()),
    'target_model_return_pct': float(comparison_row.get('return_pct', 0.0)),
    'target_model_sharpe': float(comparison_row.get('sharpe', 0.0)),
    'target_model_max_drawdown_pct': float(comparison_row.get('max_drawdown_pct', 0.0)),
    'ledger_trained': bool(ledger_row.get('trained', False)),
    'ledger_backtested': bool(ledger_row.get('backtested', False)),
    'external_references': [
        {
            'source_id': 'arxiv:1809.09401',
            'url': 'https://arxiv.org/abs/1809.09401',
            'usage': '自有摘要和结构假设；实现复用 incidence matrix、hyperedge propagation 与高阶相关建模思想，不复制论文全文或第三方代码。',
        },
        {
            'source_id': 'github:iMoonLab/HGNN',
            'url': 'https://github.com/iMoonLab/HGNN',
            'usage': '核对 HGNN 官方实现的项目边界和 hypergraph convolution 方向；本仓库未复制其代码。',
        },
    ],
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
        'qlib_risk_metrics': str((CARD_DIR / 'qlib_risk_metrics.csv').relative_to(REPO_ROOT)),
        'model_effect_deltas': str((CARD_DIR / 'model_effect_deltas.csv').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.hypergraph_native_adapter_v1',
 'status': 'completed',
 'model': 'hypergraph_mse',
 'adapter': 'hgnn_hypergraph_convolution_mse_v1',
 'estimator': 'HypergraphGraphRegressor',
 'node_count': 8,
 'feature_count': 8,
 'hyperedge_count': 11,
 'hidden_size': 8,
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 25,
 'full_matrix_proxy_model_count': 7,
 'matrix_card': 'research/hypergraph_native_adapter_v1/matrix_card',
 'matrix_models': ['gcn_mse', 'hypergraph_mse', 'ridge_mse', 'thgnn_mse'],
 'target_model_return_pct': 27.91584530515927,
 'target_model_sharpe': 2.505647454102708,
 'target_model_max_drawdown_pct': 30.878641940588956,
 'ledger_trained': True,
 'ledger_backtested': True,
 'external_references': [{'source_id': 'arxiv:1809.09401',
   'url': 'https://arxiv.org/abs/1809.09401',
   'usage': '自有摘要和结构假设；实现复用 incidence matrix、hyperedge propagation 与高阶相关建模思想，不复制论文全文或第三方代码。'},
  {'source_id': 'github:iMoonLab/HGNN',
  

## 外部参考摘要

- Feng et al., `Hypergraph Neural Networks`：论文把复杂/高阶相关关系放入 hypergraph 结构，并用 hyperedge convolution 做 representation learning。参考：https://arxiv.org/abs/1809.09401
- iMoonLab/HGNN：官方项目说明 HGNN 是 AAAI 2019 的 hypergraph neural network 代码与数据示例。参考：https://github.com/iMoonLab/HGNN
- 本仓库实现：SOXL 单资产监督任务当前没有真实股票池超边，本轮先把特征列视为 feature nodes，用全局、分组、相邻特征构造 incidence matrix，再用归一化 hypergraph propagation 做节点消息传播。后续若接入多资产/行业/供应链/资金流网络，应替换 hyperedge provider，而不是在 estimator 里硬编码实验超边。


## 中文结论

- 做了什么：在 `src/model.graph` 增加 `HypergraphGraphRegressor`，把 `hypergraph_mse` 从 `GraphFeatureRidgeRegressor(graph_kind='hypergraph')` proxy 升级为 `native_graph`。实现使用 feature-node incidence matrix、HGNN-style normalized propagation、两层 hypergraph message passing 和 pooled regression head。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`gcn_mse`、`hypergraph_mse`、`thgnn_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=25、proxy=7；`hypergraph_mse` 已进入 `native_graph`，adapter 为 `hgnn_hypergraph_convolution_mse_v1`。
- Regime/区间差异：本轮继续复用 `src/analysis` 的 window/regime/qlib 风险表，具体 winner/loser 以 `matrix_card/analysis_summary.json`、`model_review_briefing.csv` 和 `model_effect_deltas.csv` 为准。
- regime 差异：本 notebook 不额外引入新的 regime 结论；如有分层结果，以已写出的 analysis/regime/window sidecar 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：当前 hyperedge 来自特征分组和相邻结构，不是真实多资产关系；真实支持宇宙、行业/ETF 成分、资金流网络和更长 walk-forward 需要后续验证。
- 下一步：刷新全模型 ledger 和 market-regime analysis，使 native/proxy 计数、adapter 名称和模型回测记录都同步到最新框架。
